In [9]:
#Import libraries
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

#Using sqlite3 for a portable, self-contained db environment
connect_sql = sqlite3.connect('FX_data.db')
# eed ensures the synthetic data patterns remain consistent
np.random.seed(42)

#Simulating a diverse Fintech portfolio across key markets (EMEA, AMER, APAC)
num_customers = 100
customers = pd.DataFrame({
    "Customer_id": range(1, num_customers + 1),
    "Name": [f'Client {i}' for i in range(1, num_customers + 1)],
    "Country": np.random.choice(["Ireland", "United Kingdom", "United States of America", "Germany", "France", "Japan"], num_customers),
    "Industry": np.random.choice(["Retail", "Manufacturing", "Healthcare", "Tech/IT"], num_customers, p=[0.4, 0.3, 0.1, 0.2])
})
customers.to_sql("Customers", connect_sql, if_exists="replace", index=False)

#FX Rate Simulation with Market Gaps, generating 120 days of exchange rates. 
#Rates are only generated for weekdays (Mon-Fri) to simulate real-world market closures.
dates = [datetime(2026, 1, 1) + timedelta(days=i) for i in range(120)]
fx_data = []
for day in dates:
    if day.weekday() < 5:  # Monday to Friday
        # USD and GBP rates against EUR
        fx_data.append([day.strftime('%Y-%m-%d'), "USD", 0.87 + np.random.normal(0, 0.01)])
        fx_data.append([day.strftime('%Y-%m-%d'), "GBP", 1.16 + np.random.normal(0, 0.01)])
        #For Japan, using a lower standard deviation (0.0001) to keep JPY fluctuations proportional to its value
        fx_data.append([day.strftime('%Y-%m-%d'), "JPY", 0.0062 + np.random.normal(0, 0.0001)])

fx_rates = pd.DataFrame(fx_data, columns=["Rate_date", "Currency", "Rate_to_EUR"])
fx_rates.to_sql("fx_rates", connect_sql, if_exists="replace", index=False)

#Transaction Generation
transactions = []
for i in range(1, 1001):
    #Customer selection using a weighted distribution to simulate high and low volume users
    cust_id = np.random.choice(customers['Customer_id'], p=(np.arange(100, 0, -1) / np.sum(np.arange(100, 0, -1))))
    date = datetime(2026, 1, 1) + timedelta(days=np.random.randint(0, 120))
    currency = np.random.choice(["EUR", "USD", "GBP", "JPY"], p=[0.40, 0.20, 0.25, 0.15])
    
    #Base amount generation
    amt = np.random.uniform(10, 2000) if i % 2 == 0 else np.random.uniform(2000, 10000)
    
    #Japan amounts are scaled by 160x to reflect realistic nominal values and preventing having los transactions that when converted are 5 euros.
    if currency == "JPY":
        amt = amt * 160 
    #add it to transactions list
    transactions.append([i, int(cust_id), date.strftime('%Y-%m-%d'), currency, round(amt, 2)])

#Converting the generated list into a DataFrame for structured handling
trans = pd.DataFrame(transactions, columns=["Tx_id", "Customer_id", "Transaction_date", "Currency", "Amount"])
trans.to_sql("Transactions", connect_sql, if_exists="replace", index=False)#'if_exists="replace"' ensures a clean environment for each simulation run

#Close connection
print(f"Data created successfully")
connect_sql.close()

✅ Base de datos recreada. JPY ahora tiene montos realistas.
